In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
theme_park_operations


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary


In [0]:
%python
display(dbutils.fs.ls("/Volumes/workspace/default/assessment_files/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/assessment_files/water_meter_readings.csv,water_meter_readings.csv,11125,1794381987000
dbfs:/Volumes/workspace/default/assessment_files/water_reading_changes.csv,water_reading_changes.csv,956,1794381987000
dbfs:/Volumes/workspace/default/assessment_files/water_readings_incremental.csv,water_readings_incremental.csv,1391,1794381987000
dbfs:/Volumes/workspace/default/assessment_files/water_service_requests.csv,water_service_requests.csv,2662,1794381987000
dbfs:/Volumes/workspace/default/assessment_files/water_zones.json,water_zones.json,2343,1794381987000


In [0]:
%python
base_path = "/Volumes/workspace/default/assessment_files/"

readings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_meter_readings.csv")
)

display(readings_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
%python
readings_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_meter_readings_source"
)

In [0]:
%python
zones_df = (
    spark.read
    .option("multiLine", True)
    .json(base_path + "water_zones.json")
)

display(zones_df)

active,commissioned_on,daily_capacity_liters,location,zone_id,zone_name,zone_type
true,2018-01-15,240000,"List(Hyderabad, Central)",W01,Old City,Residential
true,2019-02-15,180000,"List(Hyderabad, North)",W02,Hill View,Residential
true,2020-03-15,210000,"List(Hyderabad, Central)",W03,Market District,Commercial
true,2021-04-15,280000,"List(Hyderabad, West)",W04,Lake Side,Mixed
true,2022-05-15,150000,"List(Hyderabad, East)",W05,University Quarter,Institutional
true,2023-06-15,320000,"List(Hyderabad, South)",W06,Industrial Estate,Industrial
true,2024-07-15,190000,"List(Hyderabad, North)",W07,Garden Colony,Residential
true,2018-08-15,170000,"List(Hyderabad, East)",W08,Transit Hub,Commercial
true,2019-01-15,230000,"List(Hyderabad, West)",W09,Riverfront,Mixed
true,2020-02-15,260000,"List(Hyderabad, South)",W10,South Extension,Residential


In [0]:
%python
zones_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_zones_raw"
)

In [0]:
%python
requests_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_service_requests.csv")
)

display(requests_df)

request_id,meter_id,zone_id,reported_at,issue_type,priority,estimated_cost,work_status
5001,M001,W01,2026-10-02T08:00:00.000Z,Low Pressure,High,13627.01,In Progress
5002,M008,W08,2026-10-04T09:00:00.000Z,Leak Suspected,Medium,11907.95,Resolved
5003,M015,W05,2026-10-06T10:00:00.000Z,Meter Fault,Low,5205.59,Open
5004,M022,W02,2026-10-08T11:00:00.000Z,Water Quality,High,6315.27,In Progress
5005,M029,W09,2026-10-10T12:00:00.000Z,Valve Inspection,Medium,8564.44,Resolved
5006,M006,W06,2026-10-12T13:00:00.000Z,Low Pressure,Low,2465.97,Open
5007,M013,W03,2026-10-14T14:00:00.000Z,Leak Suspected,High,16135.29,In Progress
5008,M020,W10,2026-10-16T08:00:00.000Z,Meter Fault,Medium,9106.9,Resolved
5009,M027,W07,2026-10-18T09:00:00.000Z,Water Quality,Low,3112.18,Open
5010,M004,W04,2026-10-20T10:00:00.000Z,Valve Inspection,High,10011.1,In Progress


In [0]:
%python
requests_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_service_requests"
)

In [0]:
%python
changes_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_reading_changes.csv")
)

display(changes_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp


In [0]:
%python
incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_readings_incremental.csv")
)

display(incremental_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1161,M003,W03,2026-10-19T07:00:00.000Z,17.0,2.1,0.35,700,Suspect,Gateway
1162,M005,W05,2026-10-19T08:00:00.000Z,19.3,2.38,0.48,790,Valid,SmartMeter
1163,M007,W07,2026-10-19T09:00:00.000Z,21.6,2.66,0.61,880,Valid,Gateway
1164,M009,W09,2026-10-19T10:00:00.000Z,23.9,2.94,0.74,970,Valid,SmartMeter
1165,M011,W01,2026-10-19T11:00:00.000Z,26.2,3.22,0.87,1060,Valid,Gateway
1166,M013,W03,2026-10-19T12:00:00.000Z,28.5,3.5,0.35,1150,Valid,SmartMeter
1167,M015,W05,2026-10-20T07:00:00.000Z,30.8,2.1,0.48,1240,Valid,Gateway
1168,M017,W07,2026-10-20T08:00:00.000Z,33.1,2.38,0.61,1330,Suspect,SmartMeter
1169,M019,W09,2026-10-20T09:00:00.000Z,35.4,2.66,0.74,1420,Valid,Gateway
1170,M021,W01,2026-10-20T10:00:00.000Z,37.7,2.94,0.87,1510,Valid,SmartMeter


In [0]:
%python
incremental_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_readings_incremental_staging"
)

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,water_meter_readings_source,false
default,water_readings_incremental_staging,false
default,water_service_requests,false
default,water_zones_raw,false


In [0]:
%python
base_path = "/Volumes/workspace/default/assessment_files/"

readings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_meter_readings.csv")
)

display(readings_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
%python
readings_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_meter_readings_work"
)

In [0]:
%python
changes_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_reading_changes.csv")
)

display(changes_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp


In [0]:
%python
changes_df.write.mode("overwrite").saveAsTable(
    "workspace.default.water_reading_changes"
)

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,water_meter_readings_source,false
default,water_meter_readings_work,false
default,water_reading_changes,false
default,water_readings_incremental_staging,false
default,water_service_requests,false
default,water_zones_raw,false


**Section A**

In [0]:
%sql
SELECT *
FROM workspace.default.water_meter_readings_source
LIMIT 10;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
%sql
SELECT COUNT(*) AS total_readings
FROM workspace.default.water_meter_readings_source;

total_readings
157


In [0]:
%sql
SELECT *
FROM workspace.default.water_meter_readings_source
WHERE pressure_bar < 2.5
ORDER BY pressure_bar ASC;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1071,M011,W01,2026-10-08T07:10:00.000Z,10.9,1.8,0.8,647,Valid,SmartMeter
1064,M004,W04,2026-10-07T10:00:00.000Z,20.3,1.82,0.84,708,Valid,Gateway
1024,M024,W04,2026-10-03T10:20:00.000Z,44.5,1.83,1.25,2085,Valid,FieldApp
1116,M026,W06,2026-10-12T12:10:00.000Z,28.9,1.84,0.36,1692,Valid,SmartMeter
1133,M013,W03,2026-10-14T09:00:00.000Z,15.6,1.86,0.8,900,Valid,Gateway
1028,M028,W08,2026-10-03T14:00:00.000Z,15.0,1.9,0.94,452,Valid,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1051,M021,W01,2026-10-06T07:20:00.000Z,72.0,1.93,0.27,1818,Valid,FieldApp
1052,M022,W02,2026-10-06T08:00:00.000Z,74.0,1.94,0.21,4669,Valid,Gateway
1044,M014,W04,2026-10-05T10:10:00.000Z,79.3,1.95,1.23,2661,Valid,SmartMeter


In [0]:
%sql
SELECT *
FROM workspace.default.water_meter_readings_source
WHERE flow_lpm BETWEEN 20 AND 50
AND source_system IN ('Gateway', 'SmartMeter');

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1014,M014,W04,2026-10-02T10:10:00.000Z,22.4,4.73,0.5,1108,Valid,SmartMeter
1017,M017,W07,2026-10-02T13:10:00.000Z,28.0,3.77,1.08,1318,Valid,SmartMeter
1019,M019,W09,2026-10-02T15:00:00.000Z,23.0,4.34,0.28,1091,Valid,Gateway
1022,M022,W02,2026-10-03T08:00:00.000Z,32.5,3.35,0.6,1706,Suspect,Gateway
1032,M002,W02,2026-10-04T08:10:00.000Z,34.4,1.92,0.88,1006,Valid,SmartMeter
1047,M017,W07,2026-10-05T13:10:00.000Z,28.8,3.23,0.91,1258,Valid,SmartMeter
1049,M019,W09,2026-10-05T15:00:00.000Z,21.4,2.43,0.47,611,Valid,Gateway
1050,M020,W10,2026-10-05T16:10:00.000Z,37.2,3.91,1.02,2154,Valid,SmartMeter
1056,M026,W06,2026-10-06T12:10:00.000Z,27.2,2.69,0.68,1131,Suspect,SmartMeter


In [0]:
%sql
SELECT
    reading_status,
    COUNT(*) AS reading_count
FROM workspace.default.water_meter_readings_source
GROUP BY reading_status
ORDER BY reading_status;

reading_status,reading_count
Rejected,9
Suspect,11
Valid,137


**Section B**

In [0]:
%sql
DESCRIBE TABLE workspace.default.water_zones_raw;

col_name,data_type,comment
active,boolean,null
commissioned_on,string,null
daily_capacity_liters,bigint,null
location,struct,null
zone_id,string,null
zone_name,string,null
zone_type,string,null


In [0]:
%sql
SELECT * FROM workspace.default.water_zones_raw

active,commissioned_on,daily_capacity_liters,location,zone_id,zone_name,zone_type
true,2018-01-15,240000,"List(Hyderabad, Central)",W01,Old City,Residential
true,2019-02-15,180000,"List(Hyderabad, North)",W02,Hill View,Residential
true,2020-03-15,210000,"List(Hyderabad, Central)",W03,Market District,Commercial
true,2021-04-15,280000,"List(Hyderabad, West)",W04,Lake Side,Mixed
true,2022-05-15,150000,"List(Hyderabad, East)",W05,University Quarter,Institutional
true,2023-06-15,320000,"List(Hyderabad, South)",W06,Industrial Estate,Industrial
true,2024-07-15,190000,"List(Hyderabad, North)",W07,Garden Colony,Residential
true,2018-08-15,170000,"List(Hyderabad, East)",W08,Transit Hub,Commercial
true,2019-01-15,230000,"List(Hyderabad, West)",W09,Riverfront,Mixed
true,2020-02-15,260000,"List(Hyderabad, South)",W10,South Extension,Residential


In [0]:
%sql
SELECT r.*
FROM workspace.default.water_meter_readings_source r
LEFT ANTI JOIN workspace.default.water_zones_raw z
ON r.zone_id = z.zone_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1154,M999,W99,2026-10-16T11:00:00.000Z,24.5,3.0,0.45,1150,Valid,FieldApp
1156,M012,null,2026-10-16T12:30:00.000Z,28.0,3.1,0.52,870,Valid,Gateway


In [0]:
%sql
SELECT z.*
FROM workspace.default.water_zones_raw z
LEFT ANTI JOIN workspace.default.water_meter_readings_source r
ON z.zone_id = r.zone_id;

active,commissioned_on,daily_capacity_liters,location,zone_id,zone_name,zone_type
false,2021-03-15,125000,"List(Hyderabad, West)",W11,New Development,Mixed


In [0]:
%sql
SELECT
    zone_id,
    SUM(consumption_liters) AS total_consumption,
    AVG(pressure_bar) AS average_pressure,
    COUNT(*) AS reading_count
FROM workspace.default.water_meter_readings_source
WHERE reading_status = 'Valid'
AND consumption_liters > 0
GROUP BY zone_id;

zone_id,total_consumption,average_pressure,reading_count
W01,23107,3.423076923076923,13
W02,24120,3.3061538461538458,13
W03,35665,3.019999999999999,13
W04,20781,3.2371428571428575,14
W06,20212,3.160769230769231,13
W07,25104,3.5700000000000003,13
W08,33689,2.8657142857142857,14
W09,20894,3.5469230769230773,13
W10,27808,3.717692307692307,14
W05,28717,3.632857142857143,14


In [0]:
%sql
WITH meter_totals AS (
    SELECT
        z.location.district,
        r.meter_id,
        SUM(r.consumption_liters) AS total_consumption
    FROM workspace.default.water_meter_readings_source r
    INNER JOIN workspace.default.water_zones_raw z
        ON r.zone_id = z.zone_id
    WHERE r.reading_status = 'Valid'
    GROUP BY z.location.district, r.meter_id
)
SELECT
    district,
    meter_id,
    total_consumption,
    RANK() OVER (
        PARTITION BY district
        ORDER BY total_consumption DESC
    ) AS meter_rank
FROM meter_totals;

district,meter_id,total_consumption,meter_rank
Central,M023,17814,1
Central,M001,11416,2
Central,M003,10447,3
Central,M013,7404,4
Central,M021,7395,5
Central,M011,4296,6
East,M008,14095,1
East,M018,10660,2
East,M025,10536,3
East,M015,9807,4


**Section C**

In [0]:
%python
readings_df = spark.table("workspace.default.water_meter_readings_source")

zones_df = spark.table("workspace.default.water_zones_raw")

print("Meter Readings Schema:")
readings_df.printSchema()

print("Zone Master Schema:")
zones_df.printSchema()
readings_df = spark.table("workspace.default.water_meter_readings_source")

zones_df = spark.table("workspace.default.water_zones_raw")

print("Meter Readings Schema:")
readings_df.printSchema()

print("Zone Master Schema:")
zones_df.printSchema()

Meter Readings Schema:
root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- reading_time: timestamp (nullable = true)
 |-- flow_lpm: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- chlorine_mg_l: double (nullable = true)
 |-- consumption_liters: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_system: string (nullable = true)

Zone Master Schema:
root
 |-- active: boolean (nullable = true)
 |-- commissioned_on: string (nullable = true)
 |-- daily_capacity_liters: long (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- district: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)

Meter Readings Schema:
root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_

In [0]:
%python
from pyspark.sql.functions import col

high_flow_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("flow_lpm") >= 60)
    )
    .select(
        "reading_id",
        "meter_id",
        "zone_id",
        "flow_lpm",
        "reading_status"
    )
    .orderBy(col("flow_lpm").desc())
)

display(high_flow_df)

reading_id,meter_id,zone_id,flow_lpm,reading_status
1110,M020,W10,84.0,Valid
1106,M016,W06,83.9,Valid
1083,M023,W03,83.9,Valid
1010,M010,W10,81.3,Valid
1013,M013,W03,80.7,Valid
1078,M018,W08,80.7,Valid
1093,M003,W03,80.4,Valid
1031,M001,W01,80.1,Valid
1123,M003,W03,79.7,Valid
1044,M014,W04,79.3,Valid


In [0]:
%python
from pyspark.sql.functions import col, when

flow_category_df = readings_df.withColumn(
    "flow_category",
    when(col("flow_lpm").isNull(), "Unknown")
    .when(col("flow_lpm") >= 60, "High")
    .when(col("flow_lpm") >= 30, "Medium")
    .otherwise("Low")
)

display(flow_category_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system,flow_category
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway,Medium
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter,Medium
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp,Low
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway,Medium
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter,High
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp,Medium
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway,High
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter,High
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp,Medium
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway,High


In [0]:
%python
from pyspark.sql.functions import col, sum

top_zones_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .groupBy("zone_id")
    .agg(
        sum("consumption_liters").alias("total_consumption")
    )
    .orderBy(col("total_consumption").desc())
    .limit(3)
)

display(top_zones_df)

zone_id,total_consumption
W03,35665
W08,33689
W05,28717


In [0]:
%python
from pyspark.sql.functions import col, sum

top_zones_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .groupBy("zone_id")
    .agg(
        sum("consumption_liters").alias("total_consumption")
    )
    .orderBy(col("total_consumption").desc())
    .limit(3)
)

display(top_zones_df)

zone_id,total_consumption
W03,35665
W08,33689
W05,28717


**Section D**

In [0]:
%python
from pyspark.sql.functions import col, count

duplicate_ids_df = (
    readings_df
    .groupBy("reading_id")
    .agg(count("*").alias("duplicate_count"))
    .filter(col("duplicate_count") > 1)
)

display(duplicate_ids_df)

reading_id,duplicate_count
1044,2


In [0]:
%python
duplicate_rows_df = readings_df.join(
    duplicate_ids_df.select("reading_id"),
    on="reading_id",
    how="inner"
)

display(duplicate_rows_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1044,M014,W04,2026-10-05T10:10:00.000Z,79.3,1.95,1.23,2661,Valid,SmartMeter
1044,M014,W04,2026-10-16T13:00:00.000Z,36.0,3.2,0.8,1300,Valid,Gateway


In [0]:
%python
from pyspark.sql.functions import sum, when

missing_values_df = readings_df.agg(
    sum(when(col("pressure_bar").isNull(), 1).otherwise(0)).alias("missing_pressure"),
    sum(when(col("source_system").isNull(), 1).otherwise(0)).alias("missing_source_system")
)

display(missing_values_df)

missing_pressure,missing_source_system
1,1


In [0]:
%python
invalid_readings_df = readings_df.filter(
    (col("flow_lpm") < 0) |
    (col("consumption_liters") <= 0) |
    (col("zone_id").isNull())
)

display(invalid_readings_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1151,M004,W04,2026-10-16T08:00:00.000Z,-12.5,2.8,0.62,500,Suspect,Gateway
1152,M015,W05,2026-10-16T09:00:00.000Z,22.0,2.1,0.5,0,Valid,SmartMeter
1156,M012,null,2026-10-16T12:30:00.000Z,28.0,3.1,0.52,870,Valid,Gateway


In [0]:
%python
print("Invalid records:", invalid_readings_df.count())

Invalid records: 3


In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

clean_window = Window.partitionBy("reading_id").orderBy(
    col("reading_time").desc()
)

water_readings_clean_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("flow_lpm") > 0) &
        (col("consumption_liters") > 0) &
        col("zone_id").isNotNull()
    )
    .withColumn("row_num", row_number().over(clean_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

water_readings_clean_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.water_readings_clean"
)

display(water_readings_clean_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1011,M011,W01,2026-10-02T07:10:00.000Z,13.0,4.39,0.66,313,Valid,SmartMeter


In [0]:
%sql
SELECT COUNT(*) AS clean_reading_count
FROM workspace.default.water_readings_clean;

clean_reading_count
134


In [0]:
%sql
SELECT
    s.zone_id,
    COUNT(s.request_id) AS high_priority_requests,
    AVG(r.pressure_bar) AS average_pressure
FROM workspace.default.water_service_requests s
LEFT JOIN workspace.default.water_meter_readings_source r
    ON s.zone_id = r.zone_id
WHERE s.priority = 'High'
GROUP BY s.zone_id
ORDER BY high_priority_requests DESC;

zone_id,high_priority_requests,average_pressure
W02,30,3.3573333333333326
W01,30,3.4373333333333336
W04,17,3.2829411764705885
W06,16,3.2249999999999996
W10,16,3.7079999999999997
W05,16,3.4999999999999996
W07,15,3.3706666666666663
W08,15,2.92
W03,15,3.1779999999999995
W09,15,3.6040000000000005


**Section E**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.water_readings_work_copy
AS
SELECT *
FROM workspace.default.water_meter_readings_work;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.water_readings_work_copy;

total_records
157


In [0]:
%sql DESCRIBE TABLE workspace.default.water_readings_work_copy

col_name,data_type,comment
reading_id,int,null
meter_id,string,null
zone_id,string,null
reading_time,timestamp,null
flow_lpm,double,null
pressure_bar,double,null
chlorine_mg_l,double,null
consumption_liters,int,null
reading_status,string,null
source_system,string,null


In [0]:
%sql
INSERT INTO workspace.default.water_readings_work_copy
(reading_id, meter_id, zone_id, reading_time,
 flow_lpm, pressure_bar, consumption_liters,
 reading_status, source_system)
VALUES
(900001, 'M9001', 'W01', current_timestamp(),
 45.0, 3.2, 120.0, 'Valid', 'Gateway'),
(900002, 'M9002', 'W02', current_timestamp(),
 55.0, 3.5, 150.0, 'Valid', 'SmartMeter');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
SELECT reading_id, reading_status
FROM workspace.default.water_readings_work_copy
WHERE reading_status = 'Suspect'
LIMIT 5;

reading_id,reading_status
1005,Suspect
1022,Suspect
1039,Suspect
1056,Suspect
1073,Suspect


In [0]:
%sql
SELECT reading_id, reading_status
FROM workspace.default.water_readings_work_copy
WHERE reading_status = 'Suspect'
LIMIT 5;

reading_id,reading_status
1005,Suspect
1022,Suspect
1039,Suspect
1056,Suspect
1073,Suspect


In [0]:
%sql
UPDATE workspace.default.water_readings_work_copy
SET reading_status = 'Rejected'
WHERE reading_id = 1005;

num_affected_rows
1


In [0]:
%sql
SELECT reading_id, reading_status
FROM workspace.default.water_readings_work_copy
WHERE reading_id = 1005;

reading_id,reading_status
1005,Rejected


In [0]:
%sql
SELECT COUNT(*) AS count_before
FROM workspace.default.water_readings_work_copy;

count_before
159


In [0]:
%sql
DELETE FROM workspace.default.water_readings_work_copy
WHERE reading_status = 'Rejected';

num_affected_rows
10


In [0]:
%sql
SELECT COUNT(*) AS count_after
FROM workspace.default.water_readings_work_copy;

count_after
149


In [0]:
%python
base_path = "/Volumes/workspace/default/assessment_files/"

readings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "water_meter_readings.csv")
)

display(readings_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
%python
from pyspark.sql.functions import col, sum, avg, count

zone_summary_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .groupBy("zone_id")
    .agg(
        sum("consumption_liters").alias("total_consumption"),
        avg("pressure_bar").alias("average_pressure"),
        count("*").alias("reading_count")
    )
)

zone_summary_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.zone_consumption_summary"
)

display(zone_summary_df)

zone_id,total_consumption,average_pressure,reading_count
W06,20212,3.160769230769231,13
W10,27808,3.717692307692307,14
W07,25104,3.5700000000000003,13
W08,33689,2.8657142857142857,14
W02,24120,3.3061538461538458,13
W03,35665,3.019999999999999,13
W99,1150,3.0,1
null,870,3.1,1
W04,20781,3.2371428571428575,14
W01,23107,3.423076923076923,13


**Section F**

In [0]:
%python
from pyspark.sql.functions import col

work_df = spark.table("workspace.default.water_meter_readings_work")

changes_df = spark.table("workspace.default.water_reading_changes")

In [0]:
%python
matching_ids_df = changes_df.join(
    work_df.select("reading_id"),
    on="reading_id",
    how="inner"
)

display(matching_ids_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp


In [0]:
%python
new_ids_df = changes_df.join(
    work_df.select("reading_id"),
    on="reading_id",
    how="left_anti"
)

display(new_ids_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp
1205,M014,W04,2026-10-18T12:00:00.000Z,42.8,3.0,0.65,1700,Valid,FieldApp
1206,M017,W07,2026-10-18T13:00:00.000Z,47.5,3.1,0.7,1850,Valid,FieldApp


In [0]:
%sql
MERGE INTO workspace.default.water_meter_readings_work AS target
USING workspace.default.water_reading_changes AS source
ON target.reading_id = source.reading_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,6,0,6


In [0]:
%sql
SELECT *
FROM workspace.default.water_meter_readings_work
WHERE reading_id IN (
    SELECT reading_id
    FROM workspace.default.water_reading_changes
);

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp


In [0]:
%python
source_df = spark.table("workspace.default.water_meter_readings_source")

source_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.water_readings_scd1_copy"
)

In [0]:
%python
from delta.tables import DeltaTable

target_table = DeltaTable.forName(
    spark,
    "workspace.default.water_readings_scd1_copy"
)

source_changes = spark.table("workspace.default.water_reading_changes")

In [0]:
%python
target_table.alias("target").merge(
    source_changes.alias("source"),
    "target.reading_id = source.reading_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%python
result_df = spark.table("workspace.default.water_readings_scd1_copy")

display(result_df)
print("Total records:", result_df.count())

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1011,M011,W01,2026-10-02T07:10:00.000Z,13.0,4.39,0.66,313,Valid,SmartMeter


Total records: 163


In [0]:
%sql
DESCRIBE TABLE workspace.default.water_zones_raw;

col_name,data_type,comment
active,boolean,null
commissioned_on,string,null
daily_capacity_liters,bigint,null
location,struct,null
zone_id,string,null
zone_name,string,null
zone_type,string,null


In [0]:
%sql
SELECT *
FROM workspace.default.water_zones_raw;

active,commissioned_on,daily_capacity_liters,location,zone_id,zone_name,zone_type
true,2018-01-15,240000,"List(Hyderabad, Central)",W01,Old City,Residential
true,2019-02-15,180000,"List(Hyderabad, North)",W02,Hill View,Residential
true,2020-03-15,210000,"List(Hyderabad, Central)",W03,Market District,Commercial
true,2021-04-15,280000,"List(Hyderabad, West)",W04,Lake Side,Mixed
true,2022-05-15,150000,"List(Hyderabad, East)",W05,University Quarter,Institutional
true,2023-06-15,320000,"List(Hyderabad, South)",W06,Industrial Estate,Industrial
true,2024-07-15,190000,"List(Hyderabad, North)",W07,Garden Colony,Residential
true,2018-08-15,170000,"List(Hyderabad, East)",W08,Transit Hub,Commercial
true,2019-01-15,230000,"List(Hyderabad, West)",W09,Riverfront,Mixed
true,2020-02-15,260000,"List(Hyderabad, South)",W10,South Extension,Residential


**Section G**

In [0]:
%sql
DESCRIBE DETAIL workspace.default.water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,53f7acdd-18d3-43c4-a70c-5dc54cb0a012,workspace.default.water_meter_readings_work,null,,2026-11-11T08:20:54.706Z,2026-11-11T09:28:30.000Z,List(),List(),2,8838,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",false


In [0]:
%sql
DESCRIBE HISTORY workspace.default.water_meter_readings_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T09:28:30.000Z,75635105658499,bhavani.k0408@gmail.com,MERGE,"Map(predicate -> [""(reading_id#37291 = reading_id#37311)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4198422001845961),60dcdffb-03d0-4cec-a7f3-844eb18715fa,1111-043128-rsnm06ug-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3414, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 2521, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 885, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1583)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
0,2026-11-11T08:20:57.000Z,75635105658499,bhavani.k0408@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4198422001845961),43620540-c83e-4b4a-9340-c6df6aacdcb3,1111-043128-rsnm06ug-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT *
FROM workspace.default.water_meter_readings_work VERSION AS OF 1
WHERE reading_id = 1005;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp


In [0]:
%sql
DESCRIBE DETAIL workspace.default.water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,53f7acdd-18d3-43c4-a70c-5dc54cb0a012,workspace.default.water_meter_readings_work,null,,2026-11-11T08:20:54.706Z,2026-11-11T09:28:30.000Z,List(),List(),2,8838,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",false


In [0]:
%sql
OPTIMIZE workspace.default.water_meter_readings_work;

path,metrics
,"List(1, 2, List(5839, 5839, 5839.0, 1, 5839), List(3414, 5424, 4419.0, 2, 8838), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1794389783032, 1794389784991, 8, 1, null, List(1, 6), null, 10, 10, 511, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL workspace.default.water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,53f7acdd-18d3-43c4-a70c-5dc54cb0a012,workspace.default.water_meter_readings_work,null,,2026-11-11T08:20:54.706Z,2026-11-11T09:36:25.000Z,List(),List(),1,5839,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
VACUUM workspace.default.water_meter_readings_work DRY RUN;

path


**Section H**

In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
theme_park_operations


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,water_meter_readings_source,false
default,water_meter_readings_work,false
default,water_reading_changes,false
default,water_readings_clean,false
default,water_readings_incremental_staging,false
default,water_readings_scd1_copy,false
default,water_readings_work_copy,false
default,water_service_requests,false
default,water_zones_raw,false
default,zone_consumption_summary,false


In [0]:
%sql
DESCRIBE DETAIL workspace.default.water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,53f7acdd-18d3-43c4-a70c-5dc54cb0a012,workspace.default.water_meter_readings_work,null,,2026-11-11T08:20:54.706Z,2026-11-11T09:36:25.000Z,List(),List(),1,5839,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.valid_water_readings_view AS
SELECT *
FROM workspace.default.water_meter_readings_source
WHERE reading_status = 'Valid';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW low_pressure_readings AS
SELECT *
FROM workspace.default.water_meter_readings_source
WHERE pressure_bar < 2.5;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW high_priority_requests AS
SELECT *
FROM workspace.default.water_service_requests
WHERE priority = 'High';

In [0]:
readings_df = spark.table("workspace.default.water_meter_readings_source")

parquet_path = "/Volumes/workspace/default/assessment_files/water_readings_parquet/"

readings_df.write.mode("overwrite").parquet(parquet_path)

In [0]:
parquet_df = spark.read.parquet(parquet_path)

display(parquet_df)

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
parquet_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.water_readings_parquet_delta"
)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.water_readings_parquet_delta;

total_records
157


Section J

In [0]:
%sql CREATE OR REFRESH STREAMING TABLE water_bronze
AS
SELECT *
FROM STREAM(workspace.default.water_meter_readings_source);


message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE water_silver
AS
SELECT *
FROM STREAM(water_bronze)
WHERE reading_status = 'Valid'
  AND flow_lpm > 0
  AND consumption_liters > 0;

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW water_gold
AS
SELECT
    zone_id,
    SUM(consumption_liters) AS total_consumption,
    AVG(pressure_bar) AS average_pressure,
    COUNT(*) AS reading_count
FROM water_silver
GROUP BY zone_id;

result
The operation was successfully executed.


In [0]:
%sql
SELECT * FROM water_bronze LIMIT 10;
SELECT * FROM water_silver LIMIT 10;
SELECT * FROM water_gold;

zone_id,total_consumption,average_pressure,reading_count
W06,20212,3.160769230769231,13
W10,27808,3.717692307692307,14
W07,29994,3.416875,16
W08,33689,2.8657142857142857,14
W03,40465,3.0575,16
W02,24120,3.3061538461538458,13
W99,1150,3.0,1
null,870,3.1,1
W04,20781,3.2371428571428575,14
W05,34577,3.4477777777777776,18


In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, sum, avg, count

@dp.table(name="water_bronze_py")
def water_bronze_py():
    return spark.readStream.table(
        "workspace.default.water_meter_readings_source"
    )


@dp.table(name="water_silver_py")
def water_silver_py():
    return (
        spark.readStream.table("water_bronze_py")
        .filter(
            (col("reading_status") == "Valid") &
            (col("flow_lpm") > 0) &
            (col("consumption_liters") > 0)
        )
    )


@dp.materialized_view(name="water_gold_py")
def water_gold_py():
    return (
        spark.read.table("water_silver_py")
        .groupBy("zone_id")
        .agg(
            sum("consumption_liters").alias("total_consumption"),
            avg("pressure_bar").alias("average_pressure"),
            count("*").alias("reading_count")
        )
    )

Section K

In [0]:
%sql
SELECT *
FROM workspace.default.water_readings_incremental_staging;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1161,M003,W03,2026-10-19T07:00:00.000Z,17.0,2.1,0.35,700,Suspect,Gateway
1162,M005,W05,2026-10-19T08:00:00.000Z,19.3,2.38,0.48,790,Valid,SmartMeter
1163,M007,W07,2026-10-19T09:00:00.000Z,21.6,2.66,0.61,880,Valid,Gateway
1164,M009,W09,2026-10-19T10:00:00.000Z,23.9,2.94,0.74,970,Valid,SmartMeter
1165,M011,W01,2026-10-19T11:00:00.000Z,26.2,3.22,0.87,1060,Valid,Gateway
1166,M013,W03,2026-10-19T12:00:00.000Z,28.5,3.5,0.35,1150,Valid,SmartMeter
1167,M015,W05,2026-10-20T07:00:00.000Z,30.8,2.1,0.48,1240,Valid,Gateway
1168,M017,W07,2026-10-20T08:00:00.000Z,33.1,2.38,0.61,1330,Suspect,SmartMeter
1169,M019,W09,2026-10-20T09:00:00.000Z,35.4,2.66,0.74,1420,Valid,Gateway
1170,M021,W01,2026-10-20T10:00:00.000Z,37.7,2.94,0.87,1510,Valid,SmartMeter


In [0]:
%sql
SELECT COUNT(*) AS staging_count
FROM workspace.default.water_readings_incremental_staging;

staging_count
18


In [0]:
%sql
INSERT INTO workspace.default.water_meter_readings_source
SELECT *
FROM workspace.default.water_readings_incremental_staging;

num_affected_rows,num_inserted_rows
18,18


In [0]:
%sql
SELECT COUNT(*) AS updated_source_count
FROM workspace.default.water_meter_readings_source;

updated_source_count
175


In [0]:
from pyspark.sql.functions import col, sum

readings_df = spark.table("workspace.default.water_meter_readings_source")

zone_chart_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .groupBy("zone_id")
    .agg(sum("consumption_liters").alias("total_consumption"))
    .orderBy("zone_id")
)

display(zone_chart_df)

zone_id,total_consumption
null,870
W01,25677
W02,24120
W03,40465
W04,20781
W05,34577
W06,20212
W07,29994
W08,33689
W09,25154


In [0]:
zones_df = spark.table("workspace.default.water_zones_raw")

zones_df.printSchema()
display(zones_df)

root
 |-- active: boolean (nullable = true)
 |-- commissioned_on: string (nullable = true)
 |-- daily_capacity_liters: long (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- district: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)



active,commissioned_on,daily_capacity_liters,location,zone_id,zone_name,zone_type
true,2018-01-15,240000,"List(Hyderabad, Central)",W01,Old City,Residential
true,2019-02-15,180000,"List(Hyderabad, North)",W02,Hill View,Residential
true,2020-03-15,210000,"List(Hyderabad, Central)",W03,Market District,Commercial
true,2021-04-15,280000,"List(Hyderabad, West)",W04,Lake Side,Mixed
true,2022-05-15,150000,"List(Hyderabad, East)",W05,University Quarter,Institutional
true,2023-06-15,320000,"List(Hyderabad, South)",W06,Industrial Estate,Industrial
true,2024-07-15,190000,"List(Hyderabad, North)",W07,Garden Colony,Residential
true,2018-08-15,170000,"List(Hyderabad, East)",W08,Transit Hub,Commercial
true,2019-01-15,230000,"List(Hyderabad, West)",W09,Riverfront,Mixed
true,2020-02-15,260000,"List(Hyderabad, South)",W10,South Extension,Residential


In [0]:
from pyspark.sql.functions import col, sum, to_date

readings_df = spark.table("workspace.default.water_meter_readings_source")

daily_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .withColumn("reading_date", to_date("reading_time"))
    .groupBy("reading_date")
    .agg(sum("consumption_liters").alias("total_consumption"))
    .orderBy("reading_date")
)

display(daily_df)

reading_date,total_consumption
2026-10-01,22726
2026-10-02,13169
2026-10-03,14187
2026-10-04,19704
2026-10-05,16271
2026-10-06,19793
2026-10-07,11012
2026-10-08,14937
2026-10-09,19235
2026-10-10,16787


In [0]:
scatter_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        col("pressure_bar").isNotNull() &
        col("flow_lpm").isNotNull()
    )
    .select("pressure_bar", "flow_lpm")
)

display(scatter_df)

pressure_bar,flow_lpm
3.56,39.1
4.6,59.9
2.22,25.7
2.84,56.7
2.52,35.8
3.19,65.0
3.39,70.3
4.57,30.8
2.38,81.3
4.39,13.0


In [0]:
from pyspark.sql.functions import col, sum, avg, count

readings_df = spark.table("workspace.default.water_meter_readings_source")

zone_dashboard_df = (
    readings_df
    .filter(
        (col("reading_status") == "Valid") &
        (col("consumption_liters") > 0)
    )
    .groupBy("zone_id")
    .agg(
        sum("consumption_liters").alias("total_consumption"),
        avg("pressure_bar").alias("average_pressure"),
        avg("flow_lpm").alias("average_flow"),
        count("*").alias("reading_count")
    )
)

display(zone_dashboard_df)

zone_id,total_consumption,average_pressure,average_flow,reading_count
W06,20212,3.160769230769231,43.23076923076922,13
W10,27808,3.717692307692307,49.82142857142857,14
W07,29994,3.416875,40.625000000000014,16
W08,33689,2.8657142857142857,52.49285714285714,14
W02,24120,3.3061538461538458,44.37692307692308,13
W03,40465,3.057499999999999,55.76875,16
W99,1150,3.0,24.5,1
null,870,3.1,28.0,1
W04,20781,3.2371428571428575,39.871428571428574,14
W01,25677,3.377333333333333,42.83333333333333,15


In [0]:
requests_df = spark.table("workspace.default.water_service_requests")

request_summary_df = (
    requests_df
    .groupBy("priority")
    .agg(count("*").alias("request_count"))
)

display(request_summary_df)

priority,request_count
High,12
Medium,12
Low,12


In [0]:
quality_df = (
    readings_df
    .groupBy("reading_status")
    .agg(count("*").alias("record_count"))
)

display(quality_df)

reading_status,record_count
Valid,152
Suspect,14
Rejected,9
